In [1]:
import pandas as pd
pd.options.display.float_format = '{:,.2f}'.format


In [2]:
df_orders = pd.read_csv('../data/raw/orders.csv')

print(df_orders.columns)
print(df_orders.shape)
df_orders.head()

Index(['order_id', 'user_id', 'eval_set', 'order_number', 'order_dow',
       'order_hour_of_day', 'days_since_prior_order'],
      dtype='str')
(3421083, 7)


,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,2539329,1,prior,1,2,8,NaN
1,2398795,1,prior,2,3,7,15.00
2,473747,1,prior,3,3,12,21.00
3,2254736,1,prior,4,4,7,29.00
4,431534,1,prior,5,4,15,28.00


In [3]:
df_orders_products_train = pd.read_csv('data/raw/order_products_train.csv')
print(df_orders_products_train.columns)
print(df_orders_products_train.shape)
df_orders_products_train.head()

Index(['order_id', 'product_id', 'add_to_cart_order', 'reordered'], dtype='str')
(1384617, 4)


,order_id,product_id,add_to_cart_order,reordered
0,1,49302,1,1
1,1,11109,2,1
2,1,10246,3,0
3,1,49683,4,0
4,1,43633,5,1


In [4]:
df_products = pd.read_csv("../data/raw/products.csv")
df_products.head()

,product_id,product_name,aisle_id,department_id,prices
0,1,Chocolate Sandwich Cookies,61,19,5.80
1,2,All-Seasons Salt,104,13,9.30
2,3,Robust Golden Unsweetened Oolong Tea,94,7,4.50
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1,10.50
4,5,Green Chile Anytime Sauce,5,13,4.30


In [5]:
df_departments = pd.read_csv('data/processed/departments_t.csv')
print(df_departments.columns)
print(df_departments.shape)
df_departments.head()

Index(['department_id', 'department'], dtype='str')
(21, 2)


,department_id,department
0,1,frozen
1,2,other
2,3,bakery
3,4,produce
4,5,alcohol


In [6]:
df_aisles = pd.read_csv('data/raw/aisles.csv')
print(df_aisles.columns)
print(df_aisles.shape)
df_aisles.head()

Index(['aisle_id', 'aisle'], dtype='str')
(134, 2)


,aisle_id,aisle
0,1,prepared soups salads
1,2,specialty cheeses
2,3,energy granola bars
3,4,instant foods
4,5,marinades meat preparation


In [7]:
df_orders.head()

,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,2539329,1,prior,1,2,8,NaN
1,2398795,1,prior,2,3,7,15.00
2,473747,1,prior,3,3,12,21.00
3,2254736,1,prior,4,4,7,29.00
4,431534,1,prior,5,4,15,28.00


## The <span style ='color:orange'>orders</span> Table

**Analytical questions**

- How many total orders are there?
- How many users are represented?
- What is the distribution of eval_set?
- On which days do customers place orders most frequently?
- At what hours do customers place orders most frequently?
- Are missing values in days_since_prior_order meaningful?


In [8]:
#How many total orders are there?
total_orders = len(df_orders['order_id'])
total_orders

3421083

In [9]:
#How many users are represented?
users = len(set(df_orders['user_id']))
users

206209

In [10]:
#What is the distribution of eval_set
dist_orders = df_orders['eval_set'].value_counts(normalize=True) * 100
dist_orders

eval_set
prior   93.97
train    3.84
test     2.19
Name: proportion, dtype: float64

In [11]:
#On which days do customers place orders most frequently?
df_orders['order_dow'].value_counts()

order_dow
0    600905
1    587478
2    467260
5    453368
6    448761
3    436972
4    426339
Name: count, dtype: int64

In [12]:
#At what hours do customers place orders most frequently?
df_orders['order_hour_of_day'].value_counts()

order_hour_of_day
10    288418
11    284728
15    283639
14    283042
13    277999
12    272841
16    272553
9     257812
17    228795
18    182912
8     178201
19    140569
20    104292
7      91868
21     78109
22     61468
23     40043
6      30529
0      22758
1      12398
5       9569
2       7539
4       5527
3       5474
Name: count, dtype: int64

**Are missing values in days_since_prior_order meaningful?**
- Those show users' first orders


In [13]:
df_orders.loc[df_orders["order_number"] == 1, "days_since_prior_order"].isna().all()

np.True_

In [14]:
df_orders_train = df_orders[df_orders["eval_set"] == "train"]
df_orders_train.head()

,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order
10,1187899,1,train,11,4,8,14.00
25,1492625,2,train,15,1,11,30.00
49,2196797,5,train,5,0,11,6.00
74,525192,7,train,21,2,11,6.00
78,880375,8,train,4,1,14,10.00


## The <span style="color:orange">order_products_train</span> Table

**Analytical questions**

- How many line items are there?
- How many unique orders are represented?
- How many unique products are represented?
- What proportion of items are reordered?
- What does the distribution of add_to_cart_order look like?

In [15]:
df_orders_products_train.head(20)

,order_id,product_id,add_to_cart_order,reordered
0,1,49302,1,1
1,1,11109,2,1
2,1,10246,3,0
3,1,49683,4,0
4,1,43633,5,1
5,1,13176,6,0
6,1,47209,7,0
7,1,22035,8,1
8,36,39612,1,0
9,36,19660,2,1


In [16]:
# How many line items are there?
len(df_orders_products_train)

1384617

In [17]:
# How many unique orders are represented?
unique_orders = df_orders_products_train['order_id'].nunique()
unique_orders

131209

In [18]:
# How many unique products are represented?
unique_prod = df_orders_products_train['product_id'].nunique()
unique_prod

39123

In [19]:
# What proportion of items are reordered?

dist_reordered = df_orders_products_train['reordered'].value_counts(normalize=True) * 100
dist_reordered

reordered
1   59.86
0   40.14
Name: proportion, dtype: float64

In [20]:
# What does the distribution of add_to_cart_order look like?
dist_to_card = df_orders_products_train['add_to_cart_order'].value_counts(normalize=True) * 100
dist_to_card

add_to_cart_order
1    9.48
2    8.98
3    8.45
4    7.87
5    7.28
     ... 
76   0.00
77   0.00
78   0.00
79   0.00
80   0.00
Name: proportion, Length: 80, dtype: float64

## The <span style="color:orange">products</span> Table

**Analytical questions**

- How many products exist?
- Are product IDs unique?
- How many departments are represented in the product catalog?
- How many aisles are represented in the product catalog?
- Do some product names repeat?

In [21]:
df_products

,product_id,product_name,aisle_id,department_id,prices
0,1,Chocolate Sandwich Cookies,61,19,5.80
1,2,All-Seasons Salt,104,13,9.30
2,3,Robust Golden Unsweetened Oolong Tea,94,7,4.50
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1,10.50
4,5,Green Chile Anytime Sauce,5,13,4.30
...,...,...,...,...,...
49688,49684,"Vodka, Triple Distilled, Twist of Vanilla",124,5,5.30
49689,49685,En Croute Roast Hazelnut Cranberry,42,1,3.10
49690,49686,Artisan Baguette,112,3,7.80
49691,49687,Smartblend Healthy Metabolism Dry Cat Food,41,8,4.70


In [22]:
# How many products exist?
products_unique = df_products['product_name'].nunique() 
products_unique 

49672

In [23]:
# Are product IDs unique?
if df_products['product_id'].nunique() == len(df_products['product_id']):
    print("Yes, it's unique")
else:
    print("No, it's not unique")

No, it's not unique


In [24]:
# How many departments are represented in the product catalog?
count_departments = df_products['department_id'].nunique()
count_departments

21

In [25]:
#How many aisles are represented in the product catalog?
count_aisles = df_products['aisle_id'].nunique()
count_aisles

134

In [26]:
#Do some product names repeat?
if df_products['product_name'].nunique() == len(df_products['product_id']):
    print("Product names doesn't repeat")
else:
    print("Some product names repeat")

Some product names repeat


## The <span style = 'color : orange'> departments_t </span> Table

**Analytical questions**

- How many departments exist?
- Are department IDs unique?


In [27]:
df_departments

,department_id,department
0,1,frozen
1,2,other
2,3,bakery
3,4,produce
4,5,alcohol
5,6,international
6,7,beverages
7,8,pets
8,9,dry goods pasta
9,10,bulk


In [28]:
# How many departments exist?
count_dep = df_departments["department"].nunique()
count_dep

21

In [29]:
#Are department IDs unique?
df_departments['department_id'].is_unique
  

True

## The <span style = 'color:orange'>aisles </span>Table

**Analytical questions**

- How many aisles exist?
- Are aisle IDs unique?

In [30]:
df_aisles

,aisle_id,aisle
0,1,prepared soups salads
1,2,specialty cheeses
2,3,energy granola bars
3,4,instant foods
4,5,marinades meat preparation
...,...,...
129,130,hot cereal pancake mixes
130,131,dry pasta
131,132,beauty
132,133,muscles joints pain relief


In [31]:
# How many aisles exist?
unique_aisles = df_aisles['aisle'].nunique()
unique_aisles

134

In [32]:
# Are aisle IDs unique?
df_aisles['aisle_id'].is_unique

True

# MERGE

In [33]:
df_products_departments = pd.merge(
    df_products,
    df_departments,
    on="department_id",
    how="left"
)

df_products_departments.head()

,product_id,product_name,aisle_id,department_id,prices,department
0,1,Chocolate Sandwich Cookies,61,19,5.80,snacks
1,2,All-Seasons Salt,104,13,9.30,pantry
2,3,Robust Golden Unsweetened Oolong Tea,94,7,4.50,beverages
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1,10.50,frozen
4,5,Green Chile Anytime Sauce,5,13,4.30,pantry


In [34]:
df_products_departments_check = pd.merge(
    df_products,
    df_departments,
    on="department_id",
    how="left",
    indicator=True
)

df_products_departments_check.head()

,product_id,product_name,aisle_id,department_id,prices,department,_merge
0,1,Chocolate Sandwich Cookies,61,19,5.80,snacks,both
1,2,All-Seasons Salt,104,13,9.30,pantry,both
2,3,Robust Golden Unsweetened Oolong Tea,94,7,4.50,beverages,both
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1,10.50,frozen,both
4,5,Green Chile Anytime Sauce,5,13,4.30,pantry,both


In [35]:
df_products_departments = pd.merge(
    df_products,
    df_departments,
    on="department_id",
    how="left"
)

df_products_departments.head()

,product_id,product_name,aisle_id,department_id,prices,department
0,1,Chocolate Sandwich Cookies,61,19,5.80,snacks
1,2,All-Seasons Salt,104,13,9.30,pantry
2,3,Robust Golden Unsweetened Oolong Tea,94,7,4.50,beverages
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1,10.50,frozen
4,5,Green Chile Anytime Sauce,5,13,4.30,pantry


## Homework P1: Questions unlocked after the first merge

- How many products belong to each department?
- Which departments contain the most products?
- Are products concentrated in a small number of departments?

In [36]:
# How many products belong to each department?
df_products_departments['department'].value_counts()

department
personal care      6565
snacks             6264
pantry             5371
beverages          4365
frozen             4007
dairy eggs         3449
household          3085
canned goods       2092
dry goods pasta    1858
produce            1684
bakery             1516
deli               1322
missing            1258
international      1139
breakfast          1116
babies             1081
alcohol            1056
pets                972
meat seafood        907
other               548
bulk                 38
Name: count, dtype: int64

In [37]:
df_products_enriched = pd.merge(
    df_products_departments,
    df_aisles,
    on="aisle_id",
    how="left"
)

df_products_enriched.head()

,product_id,product_name,aisle_id,department_id,prices,department,aisle
0,1,Chocolate Sandwich Cookies,61,19,5.80,snacks,cookies cakes
1,2,All-Seasons Salt,104,13,9.30,pantry,spices seasonings
2,3,Robust Golden Unsweetened Oolong Tea,94,7,4.50,beverages,tea
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1,10.50,frozen,frozen meals
4,5,Green Chile Anytime Sauce,5,13,4.30,pantry,marinades meat preparation


In [38]:
df_products_aisles_check = pd.merge(
    df_products_departments,
    df_aisles,
    on="aisle_id",
    how="left",
    indicator=True
)

df_products_aisles_check["_merge"].value_counts()

_merge
both          49693
left_only         0
right_only        0
Name: count, dtype: int64

In [39]:
df_products_aisles_check[
    df_products_aisles_check["_merge"] == "left_only"
].head()

,product_id,product_name,aisle_id,department_id,prices,department,aisle,_merge


In [40]:
print(df_products.shape)
print(df_products_departments.shape)
print(df_products_enriched.shape)

(49693, 5)
(49693, 6)
(49693, 7)


In [41]:
df_departments["department_id"].is_unique

True

In [42]:
df_aisles["aisle_id"].is_unique

True

## Homework P2: Questions unlocked after enriching products

- How many products belong to each department?
- How many products belong to each aisle?
- Which aisles are most populated?
- Which departments span the widest variety of products?
- How are departments and aisles related?

In [43]:
# How many products belong to each department?
df_products_enriched['department'].value_counts()

department
personal care      6565
snacks             6264
pantry             5371
beverages          4365
frozen             4007
dairy eggs         3449
household          3085
canned goods       2092
dry goods pasta    1858
produce            1684
bakery             1516
deli               1322
missing            1258
international      1139
breakfast          1116
babies             1081
alcohol            1056
pets                972
meat seafood        907
other               548
bulk                 38
Name: count, dtype: int64

In [44]:
# How many products belong to each aisle?
df_products_enriched['aisle'].value_counts()

aisle
missing                         1258
candy chocolate                 1246
ice cream ice                   1091
vitamins supplements            1038
yogurt                          1026
                                ... 
frozen juice                      47
baby accessories                  44
packaged produce                  32
bulk grains rice dried goods      26
bulk dried fruits vegetables      12
Name: count, Length: 134, dtype: int64

In [45]:
# Which aisles are most populated?
df_products_enriched[df_products_enriched["aisle"] != "missing"]["aisle"].value_counts()


aisle
candy chocolate                 1246
ice cream ice                   1091
vitamins supplements            1038
yogurt                          1026
chips pretzels                   989
                                ... 
frozen juice                      47
baby accessories                  44
packaged produce                  32
bulk grains rice dried goods      26
bulk dried fruits vegetables      12
Name: count, Length: 133, dtype: int64

In [46]:
# Which departments span the widest variety of products?
df_products_enriched['department'].value_counts()

department
personal care      6565
snacks             6264
pantry             5371
beverages          4365
frozen             4007
dairy eggs         3449
household          3085
canned goods       2092
dry goods pasta    1858
produce            1684
bakery             1516
deli               1322
missing            1258
international      1139
breakfast          1116
babies             1081
alcohol            1056
pets                972
meat seafood        907
other               548
bulk                 38
Name: count, dtype: int64

In [47]:
# How are departments and aisles related?
pd.crosstab(
    df_products_enriched["department"],
    df_products_enriched["aisle"]
)

aisle,air fresheners candles,asian foods,baby accessories,baby bath body care,baby food formula,bakery desserts,baking ingredients,baking supplies decor,beauty,beers coolers,...,spreads,tea,tofu meat alternatives,tortillas flat bread,trail mix snack mix,trash bags liners,vitamins supplements,water seltzer sparkling water,white wines,yogurt
department,,,,,,,,,,,,,,,,,,,,,
alcohol,0,0,0,0,0,0,0,0,0,387,...,0,0,0,0,0,0,0,0,147,0
babies,0,0,44,132,718,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
bakery,0,0,0,0,0,297,0,0,0,0,...,0,0,0,241,0,0,0,0,0,0
beverages,0,0,0,0,0,0,0,0,0,0,...,0,894,0,0,0,0,0,344,0,0
breakfast,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
bulk,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
canned goods,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
dairy eggs,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1026
deli,0,0,0,0,0,0,0,0,0,0,...,0,0,159,0,0,0,0,0,0,0


# Merge types in brief



In [48]:
pd.merge(
    df_products,
    df_departments,
    on="department_id",
    how="inner"
).head()

,product_id,product_name,aisle_id,department_id,prices,department
0,1,Chocolate Sandwich Cookies,61,19,5.80,snacks
1,2,All-Seasons Salt,104,13,9.30,pantry
2,3,Robust Golden Unsweetened Oolong Tea,94,7,4.50,beverages
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1,10.50,frozen
4,5,Green Chile Anytime Sauce,5,13,4.30,pantry


In [49]:
pd.merge(
    df_products,
    df_departments,
    on="department_id",
    how="left"
).head()

,product_id,product_name,aisle_id,department_id,prices,department
0,1,Chocolate Sandwich Cookies,61,19,5.80,snacks
1,2,All-Seasons Salt,104,13,9.30,pantry
2,3,Robust Golden Unsweetened Oolong Tea,94,7,4.50,beverages
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1,10.50,frozen
4,5,Green Chile Anytime Sauce,5,13,4.30,pantry


In [50]:
pd.merge(
    df_products,
    df_departments,
    on="department_id",
    how="outer",
    indicator=True
).head()

,product_id,product_name,aisle_id,department_id,prices,department,_merge
0,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1,10.50,frozen,both
1,8,Cut Russet Potatoes Steam N' Mash,116,1,1.10,frozen,both
2,12,Chocolate Fudge Layer Cake,119,1,9.40,frozen,both
3,18,Pizza for One Suprema Frozen Pizza,79,1,10.60,frozen,both
4,30,"Three Cheese Ziti, Marinara with Meatballs",38,1,13.80,frozen,both


In [51]:
df_train_transactions = pd.merge(
    df_orders_train,
    df_orders_products_train,
    on="order_id",
    how="left"
)

df_train_transactions.head()

,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order,product_id,add_to_cart_order,reordered
0,1187899,1,train,11,4,8,14.00,196,1,1
1,1187899,1,train,11,4,8,14.00,25133,2,1
2,1187899,1,train,11,4,8,14.00,38928,3,1
3,1187899,1,train,11,4,8,14.00,26405,4,1
4,1187899,1,train,11,4,8,14.00,39657,5,1


In [52]:
df_train_transactions_check = pd.merge(
    df_orders_train,
    df_orders_products_train,
    on="order_id",
    how="left",
    indicator=True
)

df_train_transactions_check.head()

,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order,product_id,add_to_cart_order,reordered,_merge
0,1187899,1,train,11,4,8,14.00,196,1,1,both
1,1187899,1,train,11,4,8,14.00,25133,2,1,both
2,1187899,1,train,11,4,8,14.00,38928,3,1,both
3,1187899,1,train,11,4,8,14.00,26405,4,1,both
4,1187899,1,train,11,4,8,14.00,39657,5,1,both


In [53]:
df_train_transactions_check["_merge"].value_counts()

_merge
both          1384617
left_only           0
right_only          0
Name: count, dtype: int64

In [54]:
df_train_transactions_check[
    df_train_transactions_check["_merge"] == "left_only"
].head()

,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order,product_id,add_to_cart_order,reordered,_merge


In [55]:
print(df_orders_train.shape)
print(df_orders_products_train.shape)
print(df_train_transactions.shape)

(131209, 7)
(1384617, 4)
(1384617, 10)


In [56]:
df_train_transactions.columns

Index(['order_id', 'user_id', 'eval_set', 'order_number', 'order_dow',
       'order_hour_of_day', 'days_since_prior_order', 'product_id',
       'add_to_cart_order', 'reordered'],
      dtype='str')

# Homework P3: Questions unlocked after this merge

**After merging orders_train with order_products_train, we can now answer questions such as:**

- How many products are included in each order?
- What is the typical basket size?
- Are larger baskets more common on specific days?
- Are reordered items more common at certain times of day?
- Does basket construction differ across customer order sequence?


In [57]:
# How many products are included in each order?

num_of_prod_each_ord = df_train_transactions['order_id'].value_counts()
num_of_prod_each_ord

order_id
2813632    80
1395075    80
949182     77
2869702    76
341238     76
           ..
97844       1
1954863     1
2086546     1
1014278     1
67174       1
Name: count, Length: 131209, dtype: int64

In [58]:
# What is the typical basket size?

basket_size = df_train_transactions.groupby("order_id").size()


In [59]:
# Are larger baskets more common on specific days?

baskets_common_days = df_train_transactions.groupby("order_dow").size() / df_train_transactions["order_id"].nunique()
baskets_common_days

order_dow
0   2.47
1   1.57
2   1.22
3   1.18
4   1.18
5   1.35
6   1.58
dtype: float64

In [60]:
basket_size_by_order = (
    df_train_transactions
    .groupby(["order_id", "order_dow"])
    .size()
    .reset_index(name="basket_size")
)

In [61]:
# Are reordered items more common at certain times of day?

reordered_times_of_days = df_train_transactions.groupby('order_hour_of_day')[['reordered']].sum().sort_values(by='reordered', ascending=False)
reordered_times_of_days

,reordered
order_hour_of_day,
14,70691
11,67835
15,67788
13,67656
10,66992
12,65620
16,64907
9,58497
17,57408


In [62]:
basket_size_by_order

,order_id,order_dow,basket_size
0,1,4,8
1,36,6,8
2,38,6,9
3,96,6,7
4,98,3,49
...,...,...,...
131204,3421049,1,6
131205,3421056,2,5
131206,3421058,3,8
131207,3421063,0,4


In [63]:
df_train_analytic = pd.merge(
    df_train_transactions,
    df_products_enriched,
    on="product_id",
    how="left"
)

df_train_analytic.head()

,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order,product_id,add_to_cart_order,reordered,product_name,aisle_id,department_id,prices,department,aisle
0,1187899,1,train,11,4,8,14.00,196,1,1,Soda,77.00,7.00,9.00,beverages,soft drinks
1,1187899,1,train,11,4,8,14.00,25133,2,1,Organic String Cheese,21.00,16.00,8.60,dairy eggs,packaged cheese
2,1187899,1,train,11,4,8,14.00,38928,3,1,0% Greek Strained Yogurt,120.00,16.00,12.60,dairy eggs,yogurt
3,1187899,1,train,11,4,8,14.00,26405,4,1,XL Pick-A-Size Paper Towel Rolls,54.00,17.00,1.00,household,paper goods
4,1187899,1,train,11,4,8,14.00,39657,5,1,Milk Chocolate Almonds,45.00,19.00,6.80,snacks,candy chocolate


In [64]:
df_train_analytic_check = pd.merge(
    df_train_transactions,
    df_products_enriched,
    on="product_id",
    how="left",
    indicator=True
)

df_train_analytic_check["_merge"].value_counts()

_merge
both          1384618
left_only          88
right_only          0
Name: count, dtype: int64

In [65]:
df_train_analytic_check[
    df_train_analytic_check["_merge"] == "left_only"
].head()

,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order,product_id,add_to_cart_order,reordered,product_name,aisle_id,department_id,prices,department,aisle,_merge
10780,3009052,1629,train,37,0,13,3.00,6799,24,1,NaN,NaN,NaN,NaN,NaN,NaN,left_only
52822,195363,7955,train,12,6,14,8.00,6799,4,1,NaN,NaN,NaN,NaN,NaN,NaN,left_only
59749,831974,9000,train,11,3,13,20.00,6799,19,0,NaN,NaN,NaN,NaN,NaN,NaN,left_only
64159,206208,9640,train,16,6,11,12.00,6799,6,1,NaN,NaN,NaN,NaN,NaN,NaN,left_only
81585,472316,12168,train,10,4,12,0.00,6799,12,1,NaN,NaN,NaN,NaN,NaN,NaN,left_only


In [66]:
df_train_analytic_check.head()

,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order,product_id,add_to_cart_order,reordered,product_name,aisle_id,department_id,prices,department,aisle,_merge
0,1187899,1,train,11,4,8,14.00,196,1,1,Soda,77.00,7.00,9.00,beverages,soft drinks,both
1,1187899,1,train,11,4,8,14.00,25133,2,1,Organic String Cheese,21.00,16.00,8.60,dairy eggs,packaged cheese,both
2,1187899,1,train,11,4,8,14.00,38928,3,1,0% Greek Strained Yogurt,120.00,16.00,12.60,dairy eggs,yogurt,both
3,1187899,1,train,11,4,8,14.00,26405,4,1,XL Pick-A-Size Paper Towel Rolls,54.00,17.00,1.00,household,paper goods,both
4,1187899,1,train,11,4,8,14.00,39657,5,1,Milk Chocolate Almonds,45.00,19.00,6.80,snacks,candy chocolate,both


# Homework P4 Questions unlocked after the final merge

**After building the full analytical table, we can answer much richer questions such as:**

- Which products are purchased most often?
- Which departments dominate train orders?
- Which aisles dominate train orders?
- Which departments have the highest reorder rates?
- Which aisles appear most often in large baskets?
- At what times are certain departments purchased more frequently?

In [67]:
# Which products are purchased most often?
df_train_analytic['product_name'].value_counts()

product_name
Banana                                                         18726
Bag of Organic Bananas                                         15480
Organic Strawberries                                           10894
Organic Baby Spinach                                            9784
Large Lemon                                                     8135
                                                               ...  
Chewy Reduced Sugar Granola Bars Variety Pack                      1
Plain Flavor Probiotic Acidophilus                                 1
100% Juice, Rio Red Grapefruit                                     1
Puppy Complete Nutrition Chicken & Beef Dinner Wet Dog Food        1
Organic Aromatherapeutic Moroccan Argan Oil Set                    1
Name: count, Length: 39111, dtype: int64

In [68]:
# Which departments dominate train orders?
df_train_analytic[df_train_analytic['reordered'] == 'train']['department'].value_counts()

Series([], Name: count, dtype: int64)

In [69]:
# Which aisles dominate train orders?
df_train_analytic['aisle'].value_counts()

aisle
fresh vegetables              150609
fresh fruits                  150473
packaged vegetables fruits     78493
yogurt                         55240
packaged cheese                41699
                               ...  
kitchen supplies                 448
baby bath body care              328
baby accessories                 306
frozen juice                     294
beauty                           287
Name: count, Length: 134, dtype: int64

In [70]:
# Which departments have the highest reorder rates?
dep_highest_reorder = df_train_analytic.groupby('department').size()/df_train_analytic.groupby('department')['reordered'].sum()
dep_highest_reorder.sort_values(ascending=False)

department
personal care     2.96
pantry            2.75
international     2.63
missing           2.62
other             2.58
household         2.34
canned goods      2.05
dry goods pasta   2.05
babies            1.85
frozen            1.79
breakfast         1.75
bulk              1.73
snacks            1.72
meat seafood      1.69
alcohol           1.65
deli              1.62
pets              1.59
bakery            1.58
beverages         1.52
produce           1.50
dairy eggs        1.48
dtype: float64

In [71]:
pd.crosstab(
    df_train_analytic["department"],
    df_train_analytic["reordered"]
)

reordered,0,1
department,,
alcohol,2203,3399
babies,6857,8084
bakery,17702,30692
beverages,38960,75002
breakfast,12654,16898
bulk,573,786
canned goods,24017,22782
dairy eggs,70549,146502
deli,16924,27367


In [72]:
df_train_analytic.groupby('department').size()

department
alcohol              5602
babies              14941
bakery              48394
beverages          113962
breakfast           29552
bulk                 1359
canned goods        46799
dairy eggs         217051
deli                44291
dry goods pasta     38713
frozen             100426
household           35986
international       11902
meat seafood        30307
missing              8251
other                1795
pantry              81242
personal care       21599
pets                 4497
produce            409087
snacks             118862
dtype: int64

In [73]:
# Which aisles appear most often in large baskets?
df_train_analytic.groupby('aisle').size().sort_values(ascending=False)

aisle
fresh vegetables              150609
fresh fruits                  150473
packaged vegetables fruits     78493
yogurt                         55240
packaged cheese                41699
                               ...  
kitchen supplies                 448
baby bath body care              328
baby accessories                 306
frozen juice                     294
beauty                           287
Length: 134, dtype: int64

In [74]:
# At what times are certain departments purchased more frequently?
df_train_analytic.groupby('department')[['order_hour_of_day']].value_counts(ascending=False)

department  order_hour_of_day
alcohol     15                   621
            16                   530
            13                   520
            14                   518
            12                   494
                                ... 
snacks      1                    455
            5                    289
            2                    244
            3                    203
            4                    194
Name: count, Length: 503, dtype: int64

In [75]:
pd.crosstab(
    df_train_analytic["department"],
    df_train_analytic["order_hour_of_day"]
)

order_hour_of_day,0,1,2,3,4,5,6,7,8,9,...,14,15,16,17,18,19,20,21,22,23
department,,,,,,,,,,,,,,,,,,,,,
alcohol,33,32,5,3,2,12,34,92,161,344,...,518,621,530,443,352,220,149,85,46,41
babies,70,70,49,25,22,38,202,530,948,1257,...,1191,1184,1145,858,710,594,642,489,395,195
bakery,307,175,103,75,86,127,428,1285,2489,3356,...,4260,4056,3859,3433,2748,2023,1454,1111,904,500
beverages,693,472,285,247,231,398,1029,2812,5399,8180,...,9703,9411,8669,8069,6432,4877,3268,2549,1991,1287
breakfast,175,125,60,54,62,88,264,862,1490,2143,...,2538,2376,2270,2090,1608,1318,893,742,608,345
bulk,12,13,3,1,0,5,4,27,76,88,...,133,103,91,88,78,68,42,34,36,20
canned goods,324,150,99,100,96,128,389,1171,2120,3175,...,3981,3965,3776,3154,2508,1892,1342,1146,957,560
dairy eggs,1331,827,456,391,383,617,2012,6067,11239,15146,...,18599,17946,17292,14976,12075,9082,6482,5714,4349,2534
deli,290,197,104,80,62,109,399,1131,2030,2769,...,3756,3666,3757,3211,2570,1862,1249,1068,820,528


In [76]:
data = ['customers.csv', 'states.csv']

URL = 'https://raw.githubusercontent.com/hovhannisyan91/aca/refs/heads/main/lab/python/data/raw/'

ldf = []
for i in data: 
    df = pd.read_csv(URL+i)
    ldf.append(df)

In [77]:
df_customers = ldf[0]
df_states = ldf[1]

In [78]:
df_customers.head()

,user_id,First Name,Surnam,Gender,STATE,Age,date_joined,n_dependants,fam_status,income
0,26711,Deborah,Esquivel,Female,Missouri,48,1/1/2017,3,married,165665
1,33890,Patricia,Hart,Female,New Mexico,36,1/1/2017,0,single,59285
2,65803,Kenneth,Farley,Male,Idaho,35,1/1/2017,2,married,99568
3,125935,Michelle,Hicks,Female,Iowa,40,1/1/2017,0,single,42049
4,130797,Ann,Gilmore,Female,Maryland,26,1/1/2017,1,married,40374


In [79]:
df_states.head()

,state,region,division
0,Connecticut,Northeast,New England
1,Maine,Northeast,New England
2,Massachusetts,Northeast,New England
3,New Hampshire,Northeast,New England
4,Rhode Island,Northeast,New England


In [80]:
df_customers.rename(columns={'STATE':'state'}, inplace=True)

In [81]:
df_customers_states = df_customers.merge(df_states,on = 'state')

In [82]:
df_customers_states.head()

,user_id,First Name,Surnam,Gender,state,Age,date_joined,n_dependants,fam_status,income,region,division
0,26711,Deborah,Esquivel,Female,Missouri,48,1/1/2017,3,married,165665,Midwest,West North Central
1,33890,Patricia,Hart,Female,New Mexico,36,1/1/2017,0,single,59285,West,Mountain
2,65803,Kenneth,Farley,Male,Idaho,35,1/1/2017,2,married,99568,West,Mountain
3,125935,Michelle,Hicks,Female,Iowa,40,1/1/2017,0,single,42049,Midwest,West North Central
4,130797,Ann,Gilmore,Female,Maryland,26,1/1/2017,1,married,40374,South,South Atlantic


In [83]:
df_train_analytics_final = df_train_analytic.merge(df_customers_states, on='user_id', how='left')

In [84]:
df_train_analytics_final.drop(columns=['user_id','eval_set','product_id', 'aisle_id', 'department_id'],inplace=True)

In [85]:
print(df_train_analytics_final.shape)
df_train_analytics_final.head()

(1384706, 22)


,order_id,order_number,order_dow,order_hour_of_day,days_since_prior_order,add_to_cart_order,reordered,product_name,prices,department,...,Surnam,Gender,state,Age,date_joined,n_dependants,fam_status,income,region,division
0,1187899,11,4,8,14.00,1,1,Soda,9.00,beverages,...,Nguyen,Female,Alabama,31,2/17/2019,3,married,40423,South,East South Central
1,1187899,11,4,8,14.00,2,1,Organic String Cheese,8.60,dairy eggs,...,Nguyen,Female,Alabama,31,2/17/2019,3,married,40423,South,East South Central
2,1187899,11,4,8,14.00,3,1,0% Greek Strained Yogurt,12.60,dairy eggs,...,Nguyen,Female,Alabama,31,2/17/2019,3,married,40423,South,East South Central
3,1187899,11,4,8,14.00,4,1,XL Pick-A-Size Paper Towel Rolls,1.00,household,...,Nguyen,Female,Alabama,31,2/17/2019,3,married,40423,South,East South Central
4,1187899,11,4,8,14.00,5,1,Milk Chocolate Almonds,6.80,snacks,...,Nguyen,Female,Alabama,31,2/17/2019,3,married,40423,South,East South Central


In [86]:
df_train_analytics_final.to_parquet("data/processed/instacart.parquet", index = False)